# Efficient VLM AD Kaggle Debug Runner

Run this notebook top-to-bottom on Kaggle. Edit only the repo URL/branch cell, enable GPU, and add a Kaggle Secret named `HF_TOKEN`. Smoke mode runs with verbose debug by default.

In [ ]:
# Edit these values before running on Kaggle.
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
RUN_FULL_MINI = False
FULL_DEBUG = True

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
print({"repo": GITHUB_REPO_URL, "branch": GITHUB_BRANCH, "run_full_mini": RUN_FULL_MINI, "full_debug": FULL_DEBUG})

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
else
  cd Efficient_VLM_For_Autonomous_Driving
  git fetch origin "$GITHUB_BRANCH"
  git checkout "$GITHUB_BRANCH"
  git reset --hard "origin/$GITHUB_BRANCH"
  cd /kaggle/working
fi
cd Efficient_VLM_For_Autonomous_Driving
printf 'branch=' && git rev-parse --abbrev-ref HEAD
printf 'commit=' && git rev-parse HEAD

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python -m pip install -e .
python -m pip install pycocoevalcap || true

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN secret is empty"
print("HF_TOKEN present:", bool(os.environ["HF_TOKEN"]))

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
pwd
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD
nvidia-smi || true
python - <<'PY'
import platform, torch
print('python:', platform.python_version())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
PY

## Smoke Debug Run

Each cell is intentionally separate so failures point to the exact pipeline stage.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad inspect-data --config "$CONFIG" --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset smoke --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad debug-sample --config "$CONFIG" --split train --index 0 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset smoke --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume outputs/repvit_t5_efficient_tiny_smoke/checkpoints/align_latest.pt --max-steps 20 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3

## Debug Artifacts

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE=outputs/repvit_t5_efficient_tiny_smoke
printf '\nprepared manifest\n'
cat "$PROFILE/prepared_data/manifest.json"
printf '\ncache manifest\n'
cat "$PROFILE/cache/manifest.json"
printf '\npredictions\n'
head -5 "$PROFILE/predictions.jsonl" || true
printf '\nmetrics\n'
cat "$PROFILE/metrics.json" || true
printf '\nbenchmark\n'
cat "$PROFILE/benchmark.json" || true
printf '\ndebug files\n'
find "$PROFILE/debug" -maxdepth 2 -type f | sort || true

## Optional Full Mini Run

After smoke passes, set `RUN_FULL_MINI=True` in the first cell. Full run can take significant Kaggle GPU time.

In [ ]:
if RUN_FULL_MINI:
    import subprocess
    repo = "/kaggle/working/Efficient_VLM_For_Autonomous_Driving"
    config = "configs/repvit_t5_efficient_mini.yaml"
    dbg = ["--debug", "--debug-samples", "1"] if FULL_DEBUG else []
    commands = [
        ["python", "-m", "efficient_vlm_ad", "inspect-data", "--config", config, *dbg],
        ["python", "-m", "efficient_vlm_ad", "prepare-data", "--config", config, *dbg],
        ["python", "-m", "efficient_vlm_ad", "debug-sample", "--config", config, "--split", "train", "--index", "0", *dbg],
        ["python", "-m", "efficient_vlm_ad", "prepare-features", "--config", config, *dbg],
        ["python", "-m", "efficient_vlm_ad", "train", "--config", config, "--stage", "align", *dbg],
        ["python", "-m", "efficient_vlm_ad", "train", "--config", config, "--stage", "finetune", "--resume", "outputs/repvit_t5_efficient_mini/checkpoints/align_latest.pt", *dbg],
        ["python", "-m", "efficient_vlm_ad", "evaluate", "--config", config, "--checkpoint", "outputs/repvit_t5_efficient_mini/checkpoints/finetune_latest.pt", *dbg],
        ["python", "-m", "efficient_vlm_ad", "benchmark", "--config", config, "--checkpoint", "outputs/repvit_t5_efficient_mini/checkpoints/finetune_latest.pt", *dbg],
    ]
    for command in commands:
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=repo, check=True)
else:
    print("RUN_FULL_MINI is False; smoke debug run only.")